# TernaryBoost — Pipeline no Colab T4
**PT-BitNet + ParetoQ/ZeroQAT + Tequila + Chat em ~8 min na GPU gratuita**

**Setup:** Faça upload do diretório `ternary-boost/` completo para o Colab (zip e extraia, ou use o file browser).
O notebook espera estar DENTRO do diretório `ternary-boost/`.

In [ ]:
# Cell 1: Setup — clone repo + add source dirs to path
import sys, os, time

REPO_URL = "https://github.com/adriel007/ternary-boost.git"
ROOT = "ternary-boost"

if not os.path.exists(ROOT):
    !git clone {REPO_URL}
%cd {ROOT}

# Install deps
!pip install -q transformers safetensors datasets

# Add all src dirs
for m in ["shared", "pt_bitnet", "paretoq", "tequila", "eval", "chat"]:
    sys.path.insert(0, os.path.join(m, "src"))

import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    mem_gb = getattr(gpu, 'total_memory', getattr(gpu, 'total_mem', 0)) / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {mem_gb:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Cell 2: Choose model + config
MODEL_ID = "microsoft/phi-2"
CACHE_DIR = "/content/cache"

from chat.model_loader import load_model
from chat.config import ModelEntry

entry = ModelEntry(
    name=MODEL_ID.split("/")[-1],
    path=MODEL_ID,
    device="cuda" if torch.cuda.is_available() else "cpu",
    lambada_granularity="per_channel",
)

print(f"Model: {MODEL_ID}")
print(f"Cache: {CACHE_DIR}")
print(f"Lambada: per_channel (~6 MB / 96 layers)")

In [ ]:
# Cell 3: Run pipeline
print("Starting pipeline...\n")
t0 = time.time()
model, tokenizer = load_model(entry, cache_root=CACHE_DIR)
elapsed = time.time() - t0

from collections import Counter
types = Counter(type(m).__name__ for m in model.modules() if "Linear" in type(m).__name__)
print(f"\nPipeline done in {elapsed/60:.1f} min")
print(f"Layers: {dict(types)}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")

In [ ]:
# Cell 4: Quality test
prompts = [
    "What is the capital of France?",
    "Explain machine learning in one sentence.",
    "Write a haiku about the moon.",
    "If a train goes 60mph for 2 hours, how far does it travel?",
]

model.eval()
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(inputs.input_ids, max_new_tokens=30, temperature=0.7,
                            do_sample=True, pad_token_id=tokenizer.eos_token_id)
    elapsed = time.time() - t0
    gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    tok_count = out.shape[1] - inputs.input_ids.shape[1]
    print(f"\nQ: {prompt}")
    print(f"A: {gen.strip()[:150]}")
    print(f"   ({tok_count} tok, {elapsed:.1f}s, {tok_count/elapsed:.1f} tok/s)")

In [ ]:
# Cell 5: Interactive chat
print("CHAT (type 'quit' to exit)")
print("="*50)
conversation = []
while True:
    user = input("\nYou: ")
    if user.lower() == 'quit':
        break
    conversation.append(f"User: {user}")
    prompt_text = "\n".join(conversation[-5:]) + "\nAssistant:"
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        out = model.generate(inputs.input_ids, max_new_tokens=80, temperature=0.7,
                            do_sample=True, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Assistant: {response.strip()}")
    conversation.append(f"Assistant: {response.strip()}")